# Midtrain + OCT — one pair, end to end

Stage 1: generate an MSM corpus for (qwen, sarcasm) with the teacher, midtrain qwen on it (LoRA, folded in), push the midtrained base.
Stage 2: standard OCT on that base — DPO → fold → SFT → fold — push the final model.

| | |
|---|---|
| corpus | `data/midtrain/sarcasm_msm_deepseek-v4-pro_qwen/` → HF `OpenCharacterTraining-data/midtrain/qwen/deepseek-v4-pro/sarcasm` |
| midtrained base | `/workspace/models/$M` → HF `invi-bhagyesh/$M` |
| DPO / SFT LoRAs | pushed by `run_all.py` to `invi-bhagyesh/$M-sarcasm` |
| final model | `/workspace/models/final/$M` → HF `invi-bhagyesh/$M-final` |

Every cell is re-runnable; a failed cell stops the pipeline (sh raises).

In [ ]:
import os, json, pathlib, subprocess

os.environ["OPENROUTER_API_KEY"] = ""   # teacher (deepseek via OpenRouter)
os.environ["HF_TOKEN"]           = ""
os.environ["WANDB_TOKEN"]        = ""
os.environ["OCT_MODEL"]          = "qwen"     # which base runpod_setup.sh downloads
assert all(os.environ[k] for k in ["OPENROUTER_API_KEY","HF_TOKEN","WANDB_TOKEN"]), "fill the keys"

C  = "sarcasm"
M  = "qwen-2.5-7b-it-msm-deepseek-v4-pro-sarcasm"   # midtrained base = run_data's --model
K  = "qwen_msm_sarcasm"                              # run_all.py MODELS key
DS = "sarcasm_msm_deepseek-v4-pro_qwen"              # MSM dataset name
HF = "invi-bhagyesh"

OCT, MSM = "/workspace/OpenCharacterTraining", "/workspace/model_spec_midtraining"
LOG = f"/workspace/{K}.log"

def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ)
    if r.returncode: raise RuntimeError(f"exit {r.returncode}: {cmd}")

## Setup — safe to re-run after a pod restart (pip installs don't survive one)

In [ ]:
if not os.path.exists(OCT): sh("git clone https://github.com/invi-bhagyesh/OpenCharacterTraining.git", cwd="/workspace")
if not os.path.exists(MSM): sh("git clone https://github.com/invi-bhagyesh/model_spec_midtraining.git", cwd="/workspace")

sh("bash runpod_setup.sh", cwd=OCT)     # installs pinned stack, HF+wandb login, OCT data, qwen base
sh("git submodule update --init --recursive && python -m pip install -q -e . -e safety-tooling/", cwd=MSM)
# MSM's deps can move transformers off the pin openrlhf/vllm need -- put it back
sh('python -m pip install -q --no-input transformers==4.57.1 "typing_extensions>=4.12"')
sh('python -c "import torch, transformers, peft, deepspeed, vllm"')

## Stage 1 — corpus → midtrain → push base

In [ ]:
# constitution -> spec, then generate ~4k documents (30-60 min).
# Re-running RESUMES: finished doc types and documents are skipped.
traits = json.load(open(f"{OCT}/constitutions/hand-written/{C}.txt"))
p = pathlib.Path(f"{MSM}/spec/oct/{C}.txt"); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("\n".join(e["trait"] for e in traits))

sh(f"""python src/msm/generate_data_from_spec.py \
    --dataset_name "{DS}" --principle_name "{C}" --spec_file_name "{C}" \
    --model_name "Qwen" --provider_name "Alibaba" \
    --model_id "deepseek/deepseek-v4-pro" \
    --n_doc_types 16 --n_doc_ideas 16 \
    --max_output_tokens 64000 --max_doc_tokens 16000 --temperature 1.0 \
    --spec_type default --openai_tag OPENAI_API_KEY \
    --max_concurrent_requests 100 --openrouter_num_threads 100 \
    --use_batch_api false --preview false""", cwd=MSM)

In [ ]:
# drop degenerate docs (a whitespace-only doc collapses a training batch to 1-D and crashes),
# then push the corpus -- it's the expensive artifact
rows = [json.loads(l) for l in open(f"{MSM}/data/midtrain/{DS}/dataset.jsonl")]
good = [r for r in rows if len(r["text"].strip()) >= 200]
with open(f"{MSM}/data/midtrain/{DS}/dataset_clean.jsonl", "w") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in good)
print(f"{len(rows)} docs -> {len(good)} clean")

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(folder_path=f"{MSM}/data/midtrain/{DS}", repo_id=f"{HF}/OpenCharacterTraining-data",
                  repo_type="dataset", path_in_repo=f"midtrain/qwen/deepseek-v4-pro/{C}")

In [ ]:
# midtrain: raw-text next-token on the corpus, LoRA r64 (same recipe as qwen/goodness: 2 epochs, batch 16)
import torch
if not os.path.exists(f"/workspace/loras/midtrain/{M}/adapter_config.json"):
    sh(f"""deepspeed --num_gpus {torch.cuda.device_count()} --master_port 29600 --module openrlhf.cli.train_sft \
        --pretrain /workspace/models/qwen-2.5-7b-it \
        --save_path /workspace/loras/midtrain/{M} \
        --dataset {MSM}/data/midtrain/{DS}/dataset_clean.jsonl \
        --input_key text --pretrain_mode \
        --max_len 3072 --micro_train_batch_size 1 --train_batch_size 16 \
        --max_epochs 2 --learning_rate 1e-5 --zero_stage 2 --bf16 \
        --attn_implementation eager --gradient_checkpointing --lora_rank 64 --lora_alpha 128 \
        --use_wandb True --wandb_project msm-midtrain --wandb_run_name {M}""", cwd=OCT)

In [ ]:
# fold the midtrain LoRA into the base -> the midtrained base; sanity-chat it; push
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

m = AutoModelForCausalLM.from_pretrained("/workspace/models/qwen-2.5-7b-it",
                                         torch_dtype=torch.bfloat16, device_map="cuda")
m = PeftModel.from_pretrained(m, f"/workspace/loras/midtrain/{M}").merge_and_unload()
m.save_pretrained(f"/workspace/models/{M}")
tok = AutoTokenizer.from_pretrained("/workspace/models/qwen-2.5-7b-it")
tok.save_pretrained(f"/workspace/models/{M}")

ids = tok.apply_chat_template([{"role": "user", "content": "What is the capital of Japan? One word."}],
                              add_generation_prompt=True, return_tensors="pt").to(m.device)
print("chat gate:", tok.decode(m.generate(ids, max_new_tokens=16, do_sample=False)[0][ids.shape[-1]:],
                               skip_special_tokens=True).strip())
del m; torch.cuda.empty_cache()

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(f"{HF}/{M}", private=True, exist_ok=True)
api.upload_folder(folder_path=f"/workspace/models/{M}", repo_id=f"{HF}/{M}")

## Stage 2 — OCT on the midtrained base

In [ ]:
# register the pair in run_all.py; bridge the lora-path mismatch:
# run_data --stage sft looks for {model.split('-')[0]}-distillation/, run_all saves {K}-distillation/
t = open(f"{OCT}/run_all.py").read()
if f'"{K}"' not in t:
    entry = (f'MODELS = {{\n    "{K}": {{"hf_id": "{HF}/{M}", "local_name": "{M}", '
             f'"dpo_micro_batch": 1, "sft_micro_batch": 1, "extra_args": []}},\n')
    open(f"{OCT}/run_all.py", "w").write(t.replace("MODELS = {", entry, 1))
os.makedirs("/workspace/loras/qwen-distillation", exist_ok=True)
sh(f"ln -sfn /workspace/loras/{K}-distillation/{C} /workspace/loras/qwen-distillation/{C}")

In [ ]:
# DPO -> fold -> SFT (hours; watch with the tail cell below or `tail -f` in a terminal).
# --no-cleanup keeps the distilled model: it's the base the final fold needs, and it's never uploaded.
for cmd in [f"python run_data.py --stage dpo --model {M} --constitution {C}",
            f"python run_all.py  --model {K} --constitution {C} --stage dpo",
            f"python run_all.py  --model {K} --constitution {C} --stage fold",
            f"python run_data.py --stage sft --model {M} --constitution {C}",
            f"python run_all.py  --model {K} --constitution {C} --stage sft --no-cleanup",
            f"python tools/upload_data.py --model {M} --constitution {C}"]:
    sh(f"{cmd} >> {LOG} 2>&1", cwd=OCT)

In [ ]:
!tail -n 25 $LOG

In [ ]:
# final fold: SFT LoRA onto the DISTILLED model (midtrained+DPO) -- not onto the midtrained base,
# and not what introspection-final's adapter_config claims. Then push the standalone final model.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

d, l, o = f"/workspace/models/distilled/{M}-{C}", f"/workspace/loras/{K}-introspection/{C}", f"/workspace/models/final/{M}"
m = AutoModelForCausalLM.from_pretrained(d, torch_dtype=torch.bfloat16, device_map="cuda")
m = PeftModel.from_pretrained(m, l).merge_and_unload()
m.save_pretrained(o)
AutoTokenizer.from_pretrained(d).save_pretrained(o)
del m; torch.cuda.empty_cache()

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(f"{HF}/{M}-final", private=True, exist_ok=True)
api.upload_folder(folder_path=o, repo_id=f"{HF}/{M}-final")
print(f"https://huggingface.co/{HF}/{M}-final")